# Multi-Sequence / Multi-Class Evaluation — Phase 3 Combined

**Goal:** Run Combined P3-6 on 2–3 UA-DETRAC sequences that contain bus and van classes,  
70 frames per sequence. Results feed **§4.18** of the PFE report.

| Cell | Content | GPU? |
|------|---------|------|
| M1 | Clone / pull repo | No |
| M2 | Install dependencies | No |
| M3 | Mount Drive + copy weights + images | No |
| M4 | Scan sequences for bus/van classes | No |
| M5 | Sanity check (5 frames, 1 sequence) | Yes |
| M6 | **Main run** — Combined P3-6, 70 frames × chosen sequences | Yes |
| M7 | Per-sequence results table | No |
| M8 | Save everything to Drive | No |

**Expected runtime (T4 GPU):** ~4–5 h for 70 frames × 2 sequences (Combined config = ~9× baseline per frame).  
Each cell is **resumable**: re-run after disconnect — already-processed frames are skipped automatically.

## Cell M1 — Clone / pull repo

In [ ]:
import os

REPO_DIR = '/content/pfe-latent-attack'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/03aziz03/pfe-latent-attack.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
!git log --oneline -5

## Cell M2 — Install dependencies

In [ ]:
%pip install -q "ultralytics>=8.2,<9" lpips>=0.1.4 "torchmetrics>=1.0" diffusers transformers accelerate
import os; os.environ['WANDB_DISABLED'] = 'true'
print('Dependencies installed.')

## Cell M3 — Mount Drive + copy weights + sequence images

**Drive layout expected (same as previous runs):**
```
MyDrive/PFE/
  runs/yolov8n_detrac/best.pt          ← detector weights
  DETRAC/
    Insight-MVT_Annotation_Train/
      MVI_20011/  MVI_20032/  MVI_20034/  ...   ← sequence image folders
    DETRAC-Train-Annotations-XML/
      MVI_20011.xml  MVI_20032.xml  ...          ← annotation XMLs
```
If your Drive layout differs, adjust `DRIVE_ROOT` below.

In [ ]:
from google.colab import drive
import shutil, os
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT   = '/content/drive/MyDrive/PFE'
DETRAC_IMGS  = f'{DRIVE_ROOT}/DETRAC/Insight-MVT_Annotation_Train'
DETRAC_XML   = f'{DRIVE_ROOT}/DETRAC/DETRAC-Train-Annotations-XML'
WEIGHTS_SRC  = f'{DRIVE_ROOT}/runs/yolov8n_detrac/best.pt'

# --- copy detector weights ---
os.makedirs('runs/yolov8n_detrac', exist_ok=True)
if not os.path.exists('runs/yolov8n_detrac/best.pt'):
    shutil.copy(WEIGHTS_SRC, 'runs/yolov8n_detrac/best.pt')
    print('Weights copied.')
else:
    print('Weights already present.')

# --- verify Drive layout ---
seqs = sorted(os.listdir(DETRAC_IMGS)) if os.path.exists(DETRAC_IMGS) else []
xmls = sorted(os.listdir(DETRAC_XML))  if os.path.exists(DETRAC_XML)  else []
print(f'Sequences found in Drive: {len(seqs)}')
print(f'XML annotations found:    {len(xmls)}')
print('First 10 sequences:', seqs[:10])

## Cell M4 — Scan sequences for bus / van classes

Parses DETRAC XML annotations to find sequences with bus or van detections.  
Ranks by total bus+van instance count. Top 3 are proposed as evaluation targets.

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict

XML_DIR  = Path(DETRAC_XML)
IMGS_DIR = Path(DETRAC_IMGS)

CLASS_MAP = {'car': 0, 'bus': 1, 'van': 2, 'others': 3, 'other': 3}

seq_stats = {}  # seq_name -> {car, bus, van, others, n_frames}

for xml_path in sorted(XML_DIR.glob('*.xml')):
    seq_name = xml_path.stem
    counts = defaultdict(int)
    try:
        root = ET.parse(xml_path).getroot()
        n_frames = 0
        for frame_el in root.findall('.//frame'):
            n_frames += 1
            for target in frame_el.findall('.//target'):
                attr = target.find('attribute')
                if attr is not None:
                    vtype = attr.get('vehicle_type', 'others').lower()
                    cls = CLASS_MAP.get(vtype, 3)
                    counts[cls] += 1
        seq_stats[seq_name] = {
            'car': counts[0], 'bus': counts[1],
            'van': counts[2], 'others': counts[3],
            'n_frames': n_frames,
            'has_imgs': (IMGS_DIR / seq_name).exists(),
        }
    except Exception as e:
        print(f'  [WARN] {seq_name}: {e}')

# Sort by bus+van count descending
ranked = sorted(seq_stats.items(),
                key=lambda x: x[1]['bus'] + x[1]['van'],
                reverse=True)

print(f'\n{"Sequence":<16} {"Frames":>7} {"Car":>7} {"Bus":>7} {"Van":>7} {"Others":>7}  imgs?')
print('-' * 68)
for seq, s in ranked[:20]:  # show top 20
    marker = '  ←' if s['bus'] + s['van'] > 200 else ''
    print(f"{seq:<16} {s['n_frames']:>7} {s['car']:>7} {s['bus']:>7} {s['van']:>7} {s['others']:>7}  "
          f"{'✓' if s['has_imgs'] else '✗'}{marker}")

# Auto-select top 3 sequences with images AND bus+van > 0
SELECTED = [
    seq for seq, s in ranked
    if s['has_imgs'] and (s['bus'] + s['van']) > 0
][:3]

print(f'\n>>> Auto-selected sequences: {SELECTED}')
print('(You can override SELECTED in Cell M5/M6 if needed)')

In [ ]:
# ── Optional: override the auto-selection here ──────────────────────────────
# Uncomment and edit to force specific sequences:
# SELECTED = ['MVI_20032', 'MVI_20034', 'MVI_40141']

# Copy selected sequence images to local data/multiclass/<seq>/
import shutil
from pathlib import Path

N_FRAMES_PER_SEQ = 70   # ← adjust if needed

local_seq_dirs = {}
for seq in SELECTED:
    src_dir  = Path(DETRAC_IMGS) / seq
    dst_dir  = Path(f'data/multiclass/{seq}')
    dst_dir.mkdir(parents=True, exist_ok=True)

    all_imgs = sorted(src_dir.glob('*.jpg'))
    # subsample: take every k-th frame to spread across the sequence
    stride   = max(1, len(all_imgs) // N_FRAMES_PER_SEQ)
    selected_imgs = all_imgs[::stride][:N_FRAMES_PER_SEQ]

    copied = 0
    for img in selected_imgs:
        dst = dst_dir / img.name
        if not dst.exists():
            shutil.copy(img, dst)
            copied += 1

    local_seq_dirs[seq] = dst_dir
    print(f'{seq}: {len(selected_imgs)} frames selected, {copied} copied (already had {len(selected_imgs)-copied})')

print('\nReady to run attack.')

## Cell M5 — Sanity check (5 frames, first selected sequence)

Verifies the full pipeline on 5 frames before the long run.  
Expected: DFR > 0 on at least 2 frames, no crash. Takes ~15 min on T4.

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

SEQ_SANITY = SELECTED[0]

!python scripts/run_phase3_ablation.py \
    --configs configs/phase3_combined.yaml \
    --data    data/multiclass/{SEQ_SANITY} \
    --n_frames 5 \
    --output  results/multiclass_sanity \
    --resume

In [ ]:
import json
from pathlib import Path

s = json.load(open('results/multiclass_sanity/phase3_combined/summary.json'))
print('=== Sanity check (5 frames) ===')
print(f"  DFR_prop    : {s['mean_dfr']:+.1%}")
print(f"  DFR_strict  : {s['dfr_strict_rate']:.1%}")
print(f"  ASR         : {s['mean_asr']:.1%}")
print(f"  LPIPS       : {s['mean_lpips']:.3f}")
print(f"  N frames    : {s['n_frames']}")
print()
print('Pipeline OK — proceed to Cell M6.' if s['mean_dfr'] > 0 else 'WARNING: DFR=0 on sanity set — check weights path.')

## Cell M6 — Main run: Combined P3-6, 70 frames × all selected sequences

Runs each sequence **independently** so results can be reported per-sequence in Table 4.18.  
The `--resume` flag means you can **re-run this cell after a disconnect** — already-done frames are skipped.  

**Estimated time:** ~2–2.5 h per sequence on T4 (Combined config, 70 frames).

In [ ]:
import os, time
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

for seq in SELECTED:
    print(f'\n{'='*60}')
    print(f'  Running: {seq}')
    print(f'{'='*60}')
    t0 = time.time()

    !python scripts/run_phase3_ablation.py \
        --configs configs/phase3_combined.yaml \
        --data    data/multiclass/{seq} \
        --n_frames {N_FRAMES_PER_SEQ} \
        --output  results/multiclass/{seq} \
        --resume \
        --save_images

    elapsed = time.time() - t0
    print(f'  {seq} done in {elapsed/60:.1f} min')

## Cell M7 — Per-sequence results table

Generates the table for **§4.18** of the report.  
Also prints the class distribution for each sequence (to confirm bus/van presence).

In [ ]:
import json, math
from pathlib import Path
from collections import defaultdict
import xml.etree.ElementTree as ET

CLASS_MAP_INV = {0: 'car', 1: 'bus', 2: 'van', 3: 'others'}

# ── load results ─────────────────────────────────────────────────────────────
rows = []
for seq in SELECTED:
    summary_path = Path(f'results/multiclass/{seq}/phase3_combined/summary.json')
    if not summary_path.exists():
        print(f'[WARN] {seq}: no results yet — did M6 finish?')
        continue
    s = json.load(open(summary_path))

    # Count classes present in evaluation frames via per_frame.jsonl + annotations
    # (approximate: count classes from clean detections)
    pf_path = Path(f'results/multiclass/{seq}/phase3_combined/per_frame.jsonl')
    classes_seen = set()
    if pf_path.exists():
        # Re-run detector on 5 sample frames to get class list
        pass  # filled below

    # Get class distribution from XML annotations (fast)
    xml_path = Path(DETRAC_XML) / f'{seq}.xml'
    cls_counts = defaultdict(int)
    if xml_path.exists():
        for target in ET.parse(xml_path).getroot().findall('.//target'):
            attr = target.find('attribute')
            if attr is not None:
                vt = attr.get('vehicle_type', 'others').lower()
                cls_counts[{'car': 'car', 'bus': 'bus', 'van': 'van'}.get(vt, 'others')] += 1

    cls_str = ', '.join(
        f"{k}({v})" for k, v in sorted(cls_counts.items()) if v > 0
    )

    rows.append({
        'seq':          seq,
        'n_frames':     s['n_frames'],
        'classes':      cls_str,
        'dfr_prop':     s['mean_dfr'],
        'dfr_se':       s['se_dfr'],
        'dfr_strict':   s['dfr_strict_rate'],
        'asr':          s['mean_asr'],
        'lpips':        s['mean_lpips'],
    })

# ── print table ───────────────────────────────────────────────────────────────
print(f"{'Sequence':<14} {'N':>4}  {'Classes present':<28}  {'DFR_prop':>9}  {'DFR_strict':>10}  {'ASR':>6}  {'LPIPS':>7}")
print('-' * 90)
dfr_all, strict_all, asr_all, lpips_all = [], [], [], []
for r in rows:
    print(f"{r['seq']:<14} {r['n_frames']:>4}  {r['classes']:<28}  "
          f"{r['dfr_prop']:>+8.1%}±{r['dfr_se']:.1%}  "
          f"{r['dfr_strict']:>10.1%}  {r['asr']:>6.1%}  {r['lpips']:>7.3f}")
    dfr_all.append(r['dfr_prop'])
    strict_all.append(r['dfr_strict'])
    asr_all.append(r['asr'])
    lpips_all.append(r['lpips'])

if rows:
    import numpy as np
    print('-' * 90)
    print(f"{'AGGREGATE':<14} {sum(r['n_frames'] for r in rows):>4}  {'':28}  "
          f"{np.mean(dfr_all):>+8.1%}        "
          f"{np.mean(strict_all):>10.1%}  {np.mean(asr_all):>6.1%}  {np.mean(lpips_all):>7.3f}")

# ── save table as markdown ────────────────────────────────────────────────────
md = ['| Sequence | N | Classes | DFR_prop | DFR_strict | ASR | LPIPS |',
      '|---|---|---|---|---|---|---|']
for r in rows:
    md.append(f"| {r['seq']} | {r['n_frames']} | {r['classes']} "
              f"| {r['dfr_prop']:+.1%}±{r['dfr_se']:.1%} "
              f"| {r['dfr_strict']:.1%} | {r['asr']:.1%} | {r['lpips']:.3f} |")
if rows:
    md.append(f"| **Aggregate** | {sum(r['n_frames'] for r in rows)} | — "
              f"| **{np.mean(dfr_all):+.1%}** "
              f"| **{np.mean(strict_all):.1%}** | **{np.mean(asr_all):.1%}** | **{np.mean(lpips_all):.3f}** |")

with open('results/multiclass/table_4_18.md', 'w') as f:
    f.write('\n'.join(md) + '\n')
print('\nMarkdown table saved to results/multiclass/table_4_18.md')
print('\n' + '\n'.join(md))

## Cell M8 — Save all results to Drive

In [ ]:
import shutil, os, time
from pathlib import Path

DRIVE_PFE = '/content/drive/MyDrive/PFE'

def safe_copy(src, dst):
    Path(dst).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

# 1. Copy full results/multiclass/ tree
src_results = Path('results/multiclass')
dst_results = Path(f'{DRIVE_PFE}/results/multiclass')
if src_results.exists():
    print('Copying results/multiclass → Drive...')
    shutil.copytree(src_results, dst_results, dirs_exist_ok=True)
    print('Done.')

# 2. Copy per-frame jsonl files (lightweight, needed for LaTeX table)
for seq in SELECTED:
    pf = Path(f'results/multiclass/{seq}/phase3_combined/per_frame.jsonl')
    if pf.exists():
        dst = Path(f'{DRIVE_PFE}/results/multiclass/{seq}/phase3_combined/per_frame.jsonl')
        safe_copy(pf, dst)

# 3. Copy markdown table
tbl = Path('results/multiclass/table_4_18.md')
if tbl.exists():
    safe_copy(tbl, f'{DRIVE_PFE}/results/multiclass/table_4_18.md')

print('\nAll results saved to Drive.')
print('Download results/multiclass/ from Drive and put in D:\\minimal_research\\results\\')
print('Then add the numbers from table_4_18.md to §4.18 of the report.')

---
## Checklist after the run

1. ✅ Download `results/multiclass/table_4_18.md` from Drive
2. ✅ Copy numbers into `chapters/chapter4.tex` §4.18
3. ✅ Note the 3 sequence names actually used → add to §4.1 (dataset paragraph)
4. ✅ Update `introduction.tex` N3 sentence if aggregate DFR differs significantly from 98.9%

**Metrics to paste into §4.18 Table:**
- Per-sequence: Sequence name | N frames | Classes present | DFR_prop±SE | DFR_strict | ASR | LPIPS
- Aggregate row: mean across sequences